In [16]:
import pandas as pd
import numpy as np

In [17]:
# Lire le CSV
df = pd.read_csv("data/project_bd.csv")

In [18]:
def load_mydf():
    df = pd.read_csv("data/project_bd.csv")

    rename_dict = {
        "actual \nCountry Overshoot Day \n2018": "Overshoot Day",

        "Cropland Footprint": "Cropland_Footprint_Production",
        "Grazing Footprint": "Grazing_Footprint_Production",
        "Forest Product Footprint": "Forest_Footprint_Production",
        "Fish Footprint": "Fish_Footprint_Production",
        "Built up land": "BuiltUp_Footprint_Production",
        "Carbon Footprint": "Carbon_Footprint_Production",

        "Cropland Footprint.1": "Cropland_Footprint_Consumption",
        "Grazing Footprint.1": "Grazing_Footprint_Consumption",
        "Forest Product Footprint.1": "Forest_Footprint_Consumption",
        "Fish Footprint.1": "Fish_Footprint_Consumption",
        "Built up land.1": "BuiltUp_Footprint_Consumption",
        "Carbon Footprint.1": "Carbon_Footprint_Consumption",

        "Built up land.2": "BuiltUp_Biocapacity",
        "Total biocapacity ": "Total_Biocapacity",

        "Total Ecological Footprint (Production)": "Total_Footprint_Production",
        "Total Ecological Footprint (Consumption)": "Total_Footprint_Consumption",
    }

    df = df.rename(columns={c: rename_dict[c] for c in df.columns if c in rename_dict})

    force_text = ["Country", "Region", "Income Group", "Overshoot Day", "Quality Score"]

    def clean_numeric(s):
        if s.dtype != object:
            return s
        s = s.astype(str)
        s = s.replace(["-", "--", "", "…"], np.nan)
        s = s.str.replace(r"[,$% ]", "", regex=True)
        return pd.to_numeric(s, errors="coerce")

    for col in df.columns:
        if col not in force_text:
            df[col] = clean_numeric(df[col])

    return df

In [19]:
df = load_mydf()

In [20]:
def prepare_data():
    df = load_mydf()
# DOY
    df["Overshoot_Day_DOY"] = pd.to_datetime(
    df["Overshoot Day"], errors="coerce"
    ).dt.dayofyear

    df = df.drop(columns=["Country", "Overshoot Day"])

# INCOME
    df["Income Group"] = pd.Categorical(
    df["Income Group"],
    categories=["LI", "LM", "UM", "HI"],
    ordered=True
    )

    df["Income_Group_Code"] = (
    df["Income Group"]
    .cat.codes
    .replace(-1, np.nan)
    )

# Quality Score: ORDINALE
    df["Quality Score"] = pd.Categorical(
    df["Quality Score"],
    categories=["2A", "2B", "2C", "3A"],
    ordered=True
    )

# Region: NOMINAL
    df["Region"] = df["Region"].astype("category")

    df_imputation = df.copy()

    df_model = pd.get_dummies(
    df,
    columns=["Income Group", "Quality Score", "Region"],
    drop_first=True,
    dtype="int64"
)
    df_model = df_model.drop(columns=["Income_Group_Code"])
    return df_imputation, df_model

In [21]:
if __name__ == "__main__":
    df_imputation, df_model = prepare_data()

    print(df_imputation.head())
    print(df_model.head())


  Quality Score  SDGi  Life Expectancy    HDI  Per Capita GDP  \
0            3A  53.9           64.486  0.509      564.617383   
1            3A  71.0           78.458  0.792     5046.032611   
2            3A  70.9           76.693  0.746     4759.830054   
3            3A  50.3           60.782  0.582     3233.902767   
4            2B   NaN           76.885  0.772    15134.910090   

                      Region Income Group  Population (millions)  \
0   Middle East/Central Asia           LI              37.171898   
1               Other Europe           UM               2.882740   
2                     Africa           UM              42.228398   
3                     Africa           LM              30.809801   
4  Central America/Caribbean           HI               0.096286   

   Cropland_Footprint_Production  Grazing_Footprint_Production  \
0                       0.139393                      0.157740   
1                       0.362065                      0.195718   
2 

In [22]:
pd.set_option('display.max_columns', None)
df_model.head()

,SDGi,Life Expectancy,HDI,Per Capita GDP,Population (millions),Cropland_Footprint_Production,Grazing_Footprint_Production,Forest_Footprint_Production,Fish_Footprint_Production,BuiltUp_Footprint_Production,Carbon_Footprint_Production,Total_Footprint_Production,Cropland_Footprint_Consumption,Grazing_Footprint_Consumption,Forest_Footprint_Consumption,Fish_Footprint_Consumption,BuiltUp_Footprint_Consumption,Carbon_Footprint_Consumption,Total_Footprint_Consumption,Cropland,Grazing land,Forest land,Fishing ground,BuiltUp_Biocapacity,Total_Biocapacity,Ecological (Deficit) or Reserve,Number of Earths required,Number of Countries required,Overshoot_Day_DOY,Income Group_LM,Income Group_UM,Income Group_HI,Quality Score_2B,Quality Score_2C,Quality Score_3A,Region_Asia-Pacific,Region_Central America/Caribbean,Region_EU,Region_Middle East/Central Asia,Region_North America,Region_Other Europe,Region_South America
0,53.9,64.486,0.509,564.617383,37.171898,0.139393,0.157740,0.052034,0.000084,0.028433,0.090303,0.467987,0.269788,0.165095,0.062879,0.001060,0.028433,0.155001,0.682255,0.139393,0.157740,0.014377,0.000000,0.028433,0.339943,-0.342312,0.431091,2.006970,NaN,0,0,0,0,0,1,0,0,0,1,0,0,0
1,71.0,78.458,0.792,5046.032611,2.882740,0.362065,0.195718,0.163174,0.018087,0.042248,0.639822,1.421114,0.547458,0.195848,0.207561,0.040595,0.042248,0.852748,1.886458,0.362065,0.195718,0.310590,0.082072,0.042248,0.992693,-0.893765,1.191981,1.900343,305.0,0,1,0,0,0,1,0,0,0,0,0,1,0
2,70.9,76.693,0.746,4759.830054,42.228398,0.249653,0.108329,0.079857,0.008417,0.037283,1.200642,1.684182,0.642917,0.150201,0.163528,0.013793,0.037283,1.333187,2.340909,0.249653,0.249356,0.025333,0.007541,0.037283,0.569166,-1.771742,1.479132,4.112872,245.0,0,1,0,0,0,1,0,0,0,0,0,0,0
3,50.3,60.782,0.582,3233.902767,30.809801,0.210919,0.054546,0.086584,0.078892,0.046437,0.201850,0.679229,0.329742,0.075853,0.089537,0.091383,0.046437,0.228893,0.861845,0.210919,0.918249,0.486000,0.175160,0.046437,1.836766,0.974922,0.544567,0.469218,NaN,1,0,0,0,0,1,0,0,0,0,0,0,0
4,NaN,76.885,0.772,15134.910090,0.096286,NaN,NaN,NaN,NaN,NaN,NaN,1.910373,NaN,NaN,NaN,NaN,NaN,NaN,4.655092,NaN,NaN,NaN,NaN,NaN,0.856509,-3.798584,2.941377,5.434962,123.0,0,0,1,1,0,0,0,1,0,0,0,0,0
